# 01 - LoveDA data check and split

Attach `damifinocchiaro/loveda` with Add Input before running. It holds the official Train (2,522 tiles) and Val (1,669 tiles) splits, each with Urban and Rural folders.

The notebook syncs the code repo, checks the data, inspects the masks, and writes `train.txt`, `val.txt` and `test.txt` to `/kaggle/working/splits/`. The splits are a reduced, fixed sample of the dataset (600 / 200 / 400 tiles) so the experiments in notebook 02 train quickly.

No GPU is needed. Internet must be on for the repo sync. Run the cells top to bottom.

## 0. Sync the code repo

The first run clones the GitHub repo; every later run pulls the newest commit. After pushing new code from your machine, re-run this cell to get it here.

- The repo is cloned to `/kaggle/tmp`, which is not saved with the notebook output.
- If the repo is private, add a GitHub token as a Kaggle secret named `GITHUB_TOKEN` (Add-ons -> Secrets) and attach it to this notebook. A public repo needs no token.
- After a pull, the cell clears the repo's modules from memory. Re-run your `import` cells afterwards to load the new code; no kernel restart is needed.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ABDULAH-ab/point-supervised-remote-sensing-segmentation.git'
BRANCH = 'main'
REPO_DIR = Path('/kaggle/tmp/point-supervised-remote-sensing-segmentation')

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass  # no secret attached: fine for a public repo

auth_url = REPO_URL.replace('https://', 'https://' + token + '@') if token else REPO_URL


def git(*args):
    result = subprocess.run(['git', *args], capture_output=True, text=True)
    output = result.stdout + result.stderr
    if token:
        output = output.replace(token, '***')  # never print the token
    print(output.strip())
    if result.returncode != 0:
        raise RuntimeError('git ' + args[0] + ' failed - see the message above')


if (REPO_DIR / '.git').exists():
    git('-C', str(REPO_DIR), 'pull', '--ff-only', auth_url, BRANCH)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git('clone', '--branch', BRANCH, auth_url, str(REPO_DIR))
    git('-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL)  # do not keep the token in the clone

git('-C', str(REPO_DIR), 'log', '-1', '--oneline')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Forget modules already imported from the repo, so the next import picks up the pulled code
stale = [name for name, module in list(sys.modules.items())
         if str(getattr(module, '__file__', None) or '').startswith(str(REPO_DIR))]
for name in stale:
    del sys.modules[name]
print('Repo modules cleared for re-import:', len(stale))

In [ ]:
import os
import random

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/splits')
SEED = 42

# Split sizes. A reduced study that trains in about an hour: a fixed random part of each official split.
# Set a size to None to use everything that is available.
TRAIN_SIZE = 600   # from the 2,522 official Train tiles
VAL_SIZE = 200     # from the 1,669 official Val tiles, used to choose checkpoints
TEST_SIZE = 400    # from the official Val tiles not used for val, used for the reported numbers


def pngs(folder):
    # Sorted PNG file names in a folder
    return sorted(p.name for p in Path(folder).iterdir() if p.suffix.lower() == '.png')


def id_range(names):
    # Smallest and largest numeric file name, if the names are numbers
    try:
        ids = [int(Path(n).stem) for n in names]
        return min(ids), max(ids)
    except ValueError:
        return 'names are not numeric'


def load(path):
    return np.array(Image.open(path))

## 1. Find the folders

Looks for the four folders `Train/Urban`, `Train/Rural`, `Val/Urban` and `Val/Rural` under `/kaggle/input`. Other attached datasets are ignored.

In [ ]:
DIRS = {}
for root, dirs, _ in os.walk(INPUT):
    if 'images_png' not in dirs:
        continue
    root = Path(root)
    scene = root.name
    split = root.parent.name
    if scene in ('Urban', 'Rural') and split in ('Train', 'Val'):
        assert (split, scene) not in DIRS, 'Found two copies of ' + split + '/' + scene
        DIRS[(split, scene)] = root

KEYS = [('Train', 'Urban'), ('Train', 'Rural'), ('Val', 'Urban'), ('Val', 'Rural')]
missing = [k for k in KEYS if k not in DIRS]
assert not missing, 'Folders not found: ' + str(missing)
for key in KEYS:
    print(key, DIRS[key])

## 2. Image / mask pairing

Expected: `OK` on every line, counts of 1156, 1366, 677 and 992, and no file name used in more than one folder.

In [ ]:
NAMES = {}
all_ok = True
for key in KEYS:
    imgs = pngs(DIRS[key] / 'images_png')
    masks = pngs(DIRS[key] / 'masks_png')
    ok = imgs == masks
    all_ok = all_ok and ok
    NAMES[key] = imgs
    print('OK  ' if ok else 'FAIL', '/'.join(key).ljust(12), len(imgs), 'images,', len(masks), 'masks, id range', id_range(imgs))

total = sum(len(v) for v in NAMES.values())
unique = len(set().union(*NAMES.values()))
print()
print('Total tiles:', total, '| unique file names:', unique, '(equal means no name is reused across folders)')

## 3. Look at one image and its mask

Expected: image `(1024, 1024, 3) uint8`, mask `(1024, 1024) uint8` with values within 0-7.

In [ ]:
CLASSES = ['no-data', 'background', 'building', 'road', 'water', 'barren', 'forest', 'agriculture']

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for row, key in enumerate([('Train', 'Urban'), ('Train', 'Rural')]):
    name = NAMES[key][0]
    img = load(DIRS[key] / 'images_png' / name)
    mask = load(DIRS[key] / 'masks_png' / name)
    print('/'.join(key), name, 'image', img.shape, img.dtype, '| mask', mask.shape, mask.dtype, 'values', np.unique(mask))
    axes[row, 0].imshow(img)
    axes[row, 0].set_title('/'.join(key) + ' ' + name)
    axes[row, 1].imshow(mask, vmin=0, vmax=7, cmap='tab10', interpolation='nearest')
    axes[row, 1].set_title('mask')
    axes[row, 0].axis('off')
    axes[row, 1].axis('off')
plt.tight_layout()
plt.show()

## 4. Class frequencies

Pixel share of each class over a random sample of 300 Train masks. Keep this table for the report.

In [ ]:
all_train = [(key, n) for key in KEYS[:2] for n in NAMES[key]]
sample = random.Random(SEED).sample(all_train, 300)

counts = np.zeros(256, dtype=np.int64)
for key, name in sample:
    counts += np.bincount(load(DIRS[key] / 'masks_png' / name).ravel(), minlength=256)

print('Mask values outside 0-7 (expect 0 pixels):', int(counts[8:].sum()))
print()
pixels = counts.sum()
for value, label in enumerate(CLASSES):
    print(value, label.ljust(12), '{:6.2f} %'.format(100 * counts[value] / pixels))

## 5. Write the split files

- **train** - a seeded sample of `TRAIN_SIZE` official Train tiles
- **val** - a seeded sample of `VAL_SIZE` official Val tiles
- **test** - a seeded sample of `TEST_SIZE` official Val tiles that are not in val

Every sample keeps the Urban / Rural mix of the split it comes from. No training tile comes from the official Val split, so val and test are fully separate from train.

Each line is `<Split>/<Scene>/<file>`, for example `Train/Urban/1366.png`. The image is at `<Split>/<Scene>/images_png/<file>` and the mask at `<Split>/<Scene>/masks_png/<file>` inside the dataset. In this Kaggle copy the split folder is doubled (`Train/Train/Urban/...`), so the loader should find the `<Split>/<Scene>` folders the way section 1 does instead of hard-coding the path.

In [ ]:
assert all_ok, 'Section 2 found a folder whose images and masks do not pair up'


def shares(keys, size):
    # How many tiles to take from each folder so the Urban / Rural mix is kept and the total is exact
    available = sum(len(NAMES[k]) for k in keys)
    if size is None or size > available:
        size = available
    first = round(size * len(NAMES[keys[0]]) / available)
    return {keys[0]: first, keys[1]: size - first}


rng = random.Random(SEED)
shuffled = {}
for key in KEYS:
    names = sorted(NAMES[key])
    rng.shuffle(names)
    shuffled[key] = names

train_keys, val_keys = KEYS[:2], KEYS[2:]

train_take = shares(train_keys, TRAIN_SIZE)
train = ['/'.join(k) + '/' + n for k in train_keys for n in shuffled[k][:train_take[k]]]

val_take = shares(val_keys, VAL_SIZE)
val = ['/'.join(k) + '/' + n for k in val_keys for n in shuffled[k][:val_take[k]]]

# Test tiles come from what is left of the official Val after the val tiles are taken
left = {k: shuffled[k][val_take[k]:] for k in val_keys}
left_total = sum(len(v) for v in left.values())
assert left_total > 0, 'No official Val tiles left for a test set - lower VAL_SIZE'
test_size = left_total if TEST_SIZE is None else min(TEST_SIZE, left_total)
first = min(round(test_size * len(left[val_keys[0]]) / left_total), len(left[val_keys[0]]))
test_take = {val_keys[0]: first, val_keys[1]: test_size - first}
test = ['/'.join(k) + '/' + n for k in val_keys for n in left[k][:test_take[k]]]

train, val, test = sorted(train), sorted(val), sorted(test)

assert not (set(val) & set(test)) and not (set(train) & set(val)) and not (set(train) & set(test))
assert all(e.startswith('Train/') for e in train) and all(e.startswith('Val/') for e in val + test)

OUT.mkdir(parents=True, exist_ok=True)
for split_name, entries in [('train', train), ('val', val), ('test', test)]:
    (OUT / (split_name + '.txt')).write_text('\n'.join(entries) + '\n')
    urban = sum('/Urban/' in e for e in entries)
    print(split_name.ljust(5), len(entries), 'tiles (', urban, 'urban,', len(entries) - urban, 'rural ) ->', OUT / (split_name + '.txt'))